# EDA Phase 3 — Menstrual Physiology and Hormonal Signals

**Project:** CycleInsight — Menstrual Intelligence System  
**Dataset:** `datasets/cleaned_dataset.csv`  
**Grain:** One row per day-level record; records from the same participant are not independent.  
**Scope:** Descriptive exploration of menstrual features, hormonal patterns, and temperature-signal quality. No machine learning, feature engineering, or prediction. Findings describe observed patterns only and do not imply causation.

Run cells from top to bottom. The notebook exports plots to `reports/figures/` and writes `reports/eda3_summary.md`.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = next(
    candidate.resolve()
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    if (candidate / "datasets" / "cleaned_dataset.csv").exists()
)
DATA_PATH = ROOT / "datasets" / "cleaned_dataset.csv"
FIG_DIR = ROOT / "reports" / "figures"
REPORT_PATH = ROOT / "reports" / "eda3_summary.md"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight"})

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

MENSTRUAL_CATEGORICAL = [
    "phase", "phase.1", "flow intensity", "flow_color", "Cervical mucus code",
]
MENSTRUAL_NUMERIC = [
    "bleeding_days", "inter_cycle_length", "cycle_variability",
    "day_from_last_menses", "pad_change_rate",
]
FLAG_COLS = ["heavy_bleeding_flag_proxy", "anovulation_proxy", "HMB_label"]
HORMONE_COLS = ["lh", "estrogen", "BBT", "nightly_temperature"]
TEMPERATURE_FEATURES = [
    "lag1_BBT", "lag2_BBT", "BBT_mean_7d", "BBT_std_7d", "temperature_samples",
    "baseline_relative_sample_sum", "baseline_relative_sample_sum_of_squares",
    "baseline_relative_nightly_standard_deviation",
    "baseline_relative_sample_standard_deviation",
]
REQUIRED_COLS = MENSTRUAL_CATEGORICAL + MENSTRUAL_NUMERIC + FLAG_COLS + [
    "LH_result", "lh", "estrogen", "BBT", "lag1_BBT", "lag2_BBT",
    "BBT_mean_7d", "BBT_std_7d", "nightly_temperature", "temperature_samples",
    "baseline_relative_sample_sum", "baseline_relative_sample_sum_of_squares",
    "baseline_relative_nightly_standard_deviation",
    "baseline_relative_sample_standard_deviation", "type",
]

report_tables = {}
report_findings = {}
plot_files = []


def markdown_table(table, index=True):
    """Render a DataFrame as a dependency-free Markdown table."""
    frame = table.copy()
    if index:
        frame = frame.reset_index()
    frame = frame.fillna("—").astype(str)
    frame = frame.replace({r"\|": r"\\|", r"\r?\n": " "}, regex=True)
    headers = list(frame.columns)
    rows = frame.values.tolist()
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(row) + " |" for row in rows)
    return "\n".join(lines)


def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path)
    plt.close(fig)
    plot_files.append(filename)
    print(f"Saved {path.relative_to(ROOT)}")
    return path


def numeric_summary(frame, columns):
    rows = []
    for column in columns:
        if column not in frame:
            continue
        values = pd.to_numeric(frame[column], errors="coerce")
        rows.append({
            "column": column,
            "count": int(values.count()),
            "missing": int(values.isna().sum()),
            "missing_pct": 100 * values.isna().mean(),
            "mean": values.mean(),
            "median": values.median(),
            "std": values.std(),
            "min": values.min(),
            "q1": values.quantile(0.25),
            "q3": values.quantile(0.75),
            "max": values.max(),
        })
    return pd.DataFrame(rows).set_index("column")


def iqr_outlier_table(frame, columns):
    rows = []
    for column in columns:
        if column not in frame:
            continue
        values = pd.to_numeric(frame[column], errors="coerce").dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        spread = q3 - q1
        low, high = q1 - 1.5 * spread, q3 + 1.5 * spread
        mask = (values < low) | (values > high)
        rows.append({
            "column": column, "lower_fence": low, "upper_fence": high,
            "outlier_count": int(mask.sum()), "outlier_pct": 100 * mask.mean(),
        })
    return pd.DataFrame(rows).set_index("column")


print(f"Dataset: {DATA_PATH.relative_to(ROOT)} | rows={len(df):,} | columns={df.shape[1]}")

## 1. Data Validation

Check required-column presence, inferred pandas dtypes, missingness, categorical levels, and values that fall outside transparent exploratory ranges. Range checks are data-quality prompts, not clinical diagnostic cutoffs.

In [ ]:
presence = pd.DataFrame({
    "present": [column in df.columns for column in REQUIRED_COLS],
    "dtype": [str(df[column].dtype) if column in df.columns else "MISSING" for column in REQUIRED_COLS],
    "missing": [int(df[column].isna().sum()) if column in df.columns else np.nan for column in REQUIRED_COLS],
    "missing_pct": [100 * df[column].isna().mean() if column in df.columns else np.nan for column in REQUIRED_COLS],
    "unique_non_null": [int(df[column].nunique()) if column in df.columns else np.nan for column in REQUIRED_COLS],
}, index=REQUIRED_COLS)
missing_columns = presence.index[~presence["present"]].tolist()
print(f"{len(df):,} rows | {df.shape[1]} columns | {df['id'].nunique() if 'id' in df else 'unknown'} unique participants")
display(presence)

categorical_cols = [column for column in MENSTRUAL_CATEGORICAL + ["LH_result", "type"] if column in df.columns]
category_rows = []
for column in categorical_cols:
    for value, count in df[column].value_counts(dropna=False).items():
        category_rows.append({"column": column, "category": "<missing>" if pd.isna(value) else str(value), "count": int(count), "pct": 100 * count / len(df)})
category_table = pd.DataFrame(category_rows)
display(category_table)
report_tables["validation"] = presence
report_tables["categories"] = category_table

suspicious_rules = {
    "bleeding_days": (0, 7, "expected exploratory range 0–7 days"),
    "inter_cycle_length": (21, 35, "outside broad 21–35 day audit range"),
    "day_from_last_menses": (0, None, "negative day index"),
    "pad_change_rate": (0, 1, "expected stored scale 0–1"),
    "heavy_bleeding_flag_proxy": (0, 1, "proxy outside binary 0–1 range"),
    "anovulation_proxy": (0, 1, "expected binary 0/1 encoding"),
    "HMB_label": (0, 1, "score outside 0–1 range"),
    "lh": (0, None, "negative concentration value"),
    "estrogen": (0, None, "negative concentration value"),
    "BBT": (30, 45, "outside broad temperature plausibility screen"),
    "nightly_temperature": (25, 45, "outside broad temperature plausibility screen"),
    "temperature_samples": (1, None, "non-positive sample count"),
}
suspicious_rows = []
for column, (lower, upper, rule) in suspicious_rules.items():
    values = pd.to_numeric(df[column], errors="coerce")
    mask = values.notna()
    if lower is not None:
        mask &= values < lower
    if upper is not None:
        mask |= values.notna() & (values > upper)
    flagged = values[mask]
    if len(flagged):
        suspicious_rows.append({"column": column, "rule": rule, "count": int(len(flagged)), "examples": ", ".join(map(str, flagged.head(8).tolist()))})

expected_categories = {
    "phase": {"Menstrual", "Follicular", "Fertility", "Luteal"},
    "phase.1": {"Menstrual", "Follicular", "Fertility", "Luteal"},
    "flow intensity": {"Not at all", "Spotting / Very Light", "Somewhat Light", "Light", "Moderate", "Somewhat Heavy", "Heavy", "Very Heavy"},
    "flow_color": {"Not at all", "Dark Brown / Dark Red", "Bright Red", "Yellow", "Other", "Pink", "Black", "Orange", "Grey"},
    "Cervical mucus code": {"Dry", "Creamy", "Clear", "Bloody"},
    "LH_result": {"Positive", "Negative"},
}
for column, expected in expected_categories.items():
    values = df[column].dropna().astype(str)
    odd = values[~values.isin(expected) | values.ne(values.str.strip())]
    if len(odd):
        suspicious_rows.append({"column": column, "rule": "unexpected or whitespace-padded category", "count": int(len(odd)), "examples": ", ".join(map(repr, odd.unique()[:8]))})
suspicious_table = pd.DataFrame(suspicious_rows, columns=["column", "rule", "count", "examples"])
display(suspicious_table if not suspicious_table.empty else pd.DataFrame({"result": ["No values triggered the configured exploratory checks."]}))
report_tables["suspicious"] = suspicious_table
report_findings["validation"] = (
    f"The dataset contains {len(df):,} day-level rows across {df['id'].nunique() if 'id' in df else 'an unknown number of'} participants. "
    f"{len(REQUIRED_COLS) - len(missing_columns)} of {len(REQUIRED_COLS)} requested columns are present. "
    f"{len(suspicious_table)} configured value/category checks identified at least one issue; inspect the table above. "
    "These broad screening rules are data-quality prompts, not diagnostic criteria."
)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

### Findings

Review the validation and category tables above for missing fields, sparse values, unexpected labels, and ranges that merit follow-up. The reported range checks are exploratory screens only.

## 2. Menstrual Feature Analysis

Categorical fields are summarized by record count and percentage, including missing values. Numeric fields include quartiles, histograms, boxplots, and IQR fences. IQR outliers are statistical flags, not necessarily errors.

In [ ]:
frequency_rows = []
for column in MENSTRUAL_CATEGORICAL + ["bleeding_days"]:
    counts = df[column].fillna("<missing>").astype(str).value_counts(dropna=False)
    table = pd.DataFrame({"count": counts, "pct": 100 * counts / len(df)})
    table.index.name = "category"
    frequency_rows.extend({"column": column, "category": label, "count": int(row["count"]), "pct": row["pct"]} for label, row in table.iterrows())
    fig, ax = plt.subplots(figsize=(9, max(3.2, 0.42 * len(table))))
    order = table.sort_values("count", ascending=True).index
    ax.barh(order, table.loc[order, "count"], color="#367C7A")
    ax.set(title=f"{column}: record frequency", xlabel="Records", ylabel="Category")
    save_fig(fig, f"eda3_menstrual_category_{column.lower().replace(' ', '_').replace('.', '_')}.png")

frequency_table = pd.DataFrame(frequency_rows)
display(frequency_table.round({"pct": 2}))
report_tables["menstrual_categories"] = frequency_table

menstrual_numeric_summary = numeric_summary(df, MENSTRUAL_NUMERIC)
menstrual_iqr = iqr_outlier_table(df, MENSTRUAL_NUMERIC)
display(menstrual_numeric_summary.round(3))
display(menstrual_iqr.round(3))
report_tables["menstrual_numeric_summary"] = menstrual_numeric_summary
report_tables["menstrual_iqr"] = menstrual_iqr

for column in MENSTRUAL_NUMERIC:
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [3, 1]})
    sns.histplot(values, bins="auto", ax=axes[0], color="#367C7A")
    axes[0].set(title=f"{column}: distribution", xlabel=column, ylabel="Records")
    sns.boxplot(x=values, ax=axes[1], color="#E9A23B")
    axes[1].set(title="Boxplot", xlabel=column)
    save_fig(fig, f"eda3_menstrual_numeric_{column}.png")

phase_counts = df["phase"].value_counts(dropna=False)
leading_phase = phase_counts.index[0]
leading_phase_pct = 100 * phase_counts.iloc[0] / len(df)
most_outliers = menstrual_iqr["outlier_count"].idxmax()
report_findings["menstrual"] = (
    f"The most frequent phase label is {leading_phase} ({phase_counts.iloc[0]:,} records; {leading_phase_pct:.1f}%). "
    f"Among the numeric menstrual measures, {most_outliers} has the most IQR-flagged records "
    f"({int(menstrual_iqr.loc[most_outliers, 'outlier_count']):,}); review its distribution and the fences before interpreting extremes."
)
print(report_findings["menstrual"])

### Findings

Use the frequency table to see dominant menstrual labels and the numeric summaries to compare typical values with their spread. IQR outliers are distribution-relative and should be reviewed in context rather than automatically removed.

## 3. Flag Analysis

Report raw value counts for all three proxy fields. Prevalence is meaningful only for established binary encodings; continuous scores are not dichotomized without a documented threshold.

In [ ]:
flag_rows = []
flag_frequency_rows = []
for column in FLAG_COLS:
    values = pd.to_numeric(df[column], errors="coerce")
    observed = values.dropna()
    row = {
        "column": column,
        "encoding_note": "binary" if set(observed.unique()).issubset({0, 1}) else "continuous/non-binary; no cutoff applied",
        "count": int(observed.size),
        "missing": int(values.isna().sum()),
        "unique_values": int(observed.nunique()),
        "nonzero_count": int((observed != 0).sum()),
        "nonzero_pct": 100 * (observed != 0).mean() if len(observed) else np.nan,
        "min": observed.min(), "median": observed.median(), "max": observed.max(),
    }
    flag_rows.append(row)
    counts = values.value_counts(dropna=False).sort_index()
    for value, count in counts.items():
        flag_frequency_rows.append({
            "column": column,
            "stored_value": "<missing>" if pd.isna(value) else str(value),
            "count": int(count),
            "pct_of_records": 100 * count / len(df),
        })

flag_summary = pd.DataFrame(flag_rows).set_index("column")
flag_frequency_table = pd.DataFrame(flag_frequency_rows)
display(flag_summary.round(3))
display(flag_frequency_table.round({"pct_of_records": 3}))
report_tables["flags"] = flag_summary
report_tables["flag_frequencies"] = flag_frequency_table

for column in FLAG_COLS:
    counts = pd.to_numeric(df[column], errors="coerce").value_counts(dropna=False).sort_index()
    labels = ["<missing>" if pd.isna(value) else str(value) for value in counts.index]
    positions = np.arange(len(counts))
    fig, ax = plt.subplots(figsize=(max(7, min(15, len(counts) * 0.18)), 4))
    ax.bar(positions, counts.values, color="#C66B4E")
    if len(counts) <= 20:
        ax.set_xticks(positions, labels, rotation=35, ha="right")
    else:
        tick_positions = np.linspace(0, len(counts) - 1, 12, dtype=int)
        ax.set_xticks(tick_positions, [labels[position] for position in tick_positions], rotation=35, ha="right")
    ax.set(title=f"{column}: exact stored-value counts", xlabel="Stored value", ylabel="Records")
    save_fig(fig, f"eda3_flag_values_{column}.png")

binary_cols = [column for column in FLAG_COLS if set(pd.to_numeric(df[column], errors="coerce").dropna().unique()).issubset({0, 1})]
if binary_cols:
    binary_prevalences = {column: 100 * pd.to_numeric(df[column], errors="coerce").dropna().mean() for column in binary_cols}
    majority_shares = {column: max(value, 100 - value) for column, value in binary_prevalences.items()}
    flagged_imbalance = {column: share for column, share in majority_shares.items() if share >= 70}
    if flagged_imbalance:
        imbalance_note = "An exploratory 70% majority-class screen flags " + ", ".join(f"{column} ({share:.1f}% majority)" for column, share in flagged_imbalance.items()) + "."
    else:
        imbalance_note = "No binary field reaches a 70% majority share under this exploratory screen."
else:
    binary_prevalences = {}
    imbalance_note = "No binary flag encodings were found."
report_findings["flags"] = (
    f"{imbalance_note} Binary-field positive-class shares: "
    + (", ".join(f"{key}={value:.1f}%" for key, value in binary_prevalences.items()) or "not applicable")
    + ". Exact count/percentage frequencies are shown for all stored values. HMB_label is treated as a continuous score; no clinical threshold or prevalence class is inferred."
)
print(report_findings["flags"])

### Findings

The tables show exact stored encodings, nonzero rates, and score ranges. Class imbalance is assessed only for true 0/1 fields; `HMB_label` remains continuous unless a valid threshold is documented.

## 4. Hormonal and Primary Temperature Analysis

For `lh`, `estrogen`, `BBT`, and `nightly_temperature`, report count, missingness, mean, median, standard deviation, min/max, quartiles, histograms, KDEs, boxplots, and IQR-based outlier counts.

In [ ]:
hormone_summary = numeric_summary(df, HORMONE_COLS)
hormone_iqr = iqr_outlier_table(df, HORMONE_COLS)
display(hormone_summary.round(4))
display(hormone_iqr.round(4))
report_tables["hormone_summary"] = hormone_summary
report_tables["hormone_iqr"] = hormone_iqr

for column in HORMONE_COLS:
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw={"width_ratios": [3, 3, 1]})
    sns.histplot(values, bins="auto", ax=axes[0], color="#367C7A")
    axes[0].set(title="Histogram", xlabel=column, ylabel="Records")
    sns.kdeplot(values, ax=axes[1], color="#C66B4E", fill=True, warn_singular=False)
    axes[1].set(title="KDE (smoothed density)", xlabel=column, ylabel="Density")
    sns.boxplot(x=values, ax=axes[2], color="#E9A23B")
    axes[2].set(title="Boxplot", xlabel=column)
    fig.suptitle(f"{column}: observed distribution", y=1.03)
    save_fig(fig, f"eda3_hormone_{column.lower()}.png")

most_hormone_outliers = hormone_iqr["outlier_count"].idxmax()
report_findings["hormones"] = (
    f"Across the four primary signals, {most_hormone_outliers} has the largest IQR-flagged count "
    f"({int(hormone_iqr.loc[most_hormone_outliers, 'outlier_count']):,} records). "
    "The KDEs are smoothed views of the observed distributions; they do not establish clinical reference ranges or biological effects. "
    "Inspect the summary table for scale, spread, and missingness."
)
print(report_findings["hormones"])

### Findings

Compare observed centers, spread, missingness, and IQR flags across the signals. IQR flags identify statistical extremes only; they are not evidence of measurement error or a clinical abnormality.

## 5. Temperature Feature Analysis

Inspect lagged and rolling BBT summaries, temperature sample counts, and baseline-relative aggregates. Report their observed distributions and statistical outliers without treating derived values as independent biological measurements.

In [ ]:
temperature_summary = numeric_summary(df, TEMPERATURE_FEATURES)
temperature_iqr = iqr_outlier_table(df, TEMPERATURE_FEATURES)
display(temperature_summary.round(4))
display(temperature_iqr.round(4))
report_tables["temperature_summary"] = temperature_summary
report_tables["temperature_iqr"] = temperature_iqr

fig, axes = plt.subplots(len(TEMPERATURE_FEATURES), 2, figsize=(12, 3.1 * len(TEMPERATURE_FEATURES)))
for index, column in enumerate(TEMPERATURE_FEATURES):
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    sns.histplot(values, bins="auto", ax=axes[index, 0], color="#367C7A")
    axes[index, 0].set(title=f"{column}: distribution", xlabel=column, ylabel="Records")
    sns.boxplot(x=values, ax=axes[index, 1], color="#E9A23B")
    axes[index, 1].set(title="IQR view", xlabel=column)
fig.tight_layout()
save_fig(fig, "eda3_temperature_feature_distributions.png")

low_samples = int((pd.to_numeric(df["temperature_samples"], errors="coerce") <= 10).sum())
near_zero_spread = int((pd.to_numeric(df["BBT_std_7d"], errors="coerce") <= 0.001).sum())
constant_type = df["type"].nunique(dropna=True) == 1
report_findings["temperature"] = (
    f"`temperature_samples` ranges from {temperature_summary.loc['temperature_samples', 'min']:.0f} to "
    f"{temperature_summary.loc['temperature_samples', 'max']:.0f} (median {temperature_summary.loc['temperature_samples', 'median']:.0f}); "
    f"{low_samples} records have 10 or fewer samples. `BBT_std_7d` is at or below 0.001 in {near_zero_spread} records. "
    f"`nightly_temperature` ranges from {df['nightly_temperature'].min():.2f} to {df['nightly_temperature'].max():.2f}; "
    f"the dataset's `type` field is {'constant' if constant_type else 'not constant'} ({', '.join(map(str, df['type'].dropna().unique()[:5]))}). "
    "The scale/units and acquisition meaning should be confirmed before interpreting temperature values physiologically."
)
print(report_findings["temperature"])

### Findings

Temperature-derived measures vary substantially in scale and units. Pay particular attention to the sample-count tail, near-zero rolling variation, and the `type` metadata; confirm device and unit semantics before interpreting these as core-body physiology.

## 6. Biological Consistency Checks

Compare observed `lh`, `estrogen`, and `BBT` distributions by `phase`. Tables and plots are record-level summaries; repeated rows per participant and phase-label quality limit interpretation. No causal or inferential claim is made.

In [ ]:
BIO_COLS = ["lh", "estrogen", "BBT"]
phase_frame = df.dropna(subset=["phase"]).copy()
phase_mean = phase_frame.groupby("phase")[BIO_COLS].mean()
phase_stats = phase_frame.groupby("phase")[BIO_COLS].agg(["count", "mean", "median", "std"])
phase_stats.columns = [f"{column}_{stat}" for column, stat in phase_stats.columns]
phase_order = [value for value in ["Menstrual", "Follicular", "Fertility", "Luteal"] if value in phase_frame["phase"].unique()]
phase_order += [value for value in phase_frame["phase"].dropna().unique() if value not in phase_order]
display(phase_mean.reindex(phase_order).round(4))
display(phase_stats.reindex(phase_order).round(4))
report_tables["phase_means"] = phase_mean.reindex(phase_order)
report_tables["phase_stats"] = phase_stats.reindex(phase_order)

for column in BIO_COLS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    sns.boxplot(data=phase_frame, x="phase", y=column, order=phase_order, hue="phase", hue_order=phase_order, legend=False, ax=axes[0], palette="Set2")
    axes[0].set(title=f"{column} by phase: boxplot", xlabel="Phase", ylabel=column)
    sns.violinplot(data=phase_frame, x="phase", y=column, order=phase_order, hue="phase", hue_order=phase_order, legend=False, cut=0, inner="quartile", ax=axes[1], palette="Set2")
    axes[1].set(title=f"{column} by phase: violin", xlabel="Phase", ylabel=column)
    for ax in axes:
        ax.tick_params(axis="x", rotation=20)
    save_fig(fig, f"eda3_phase_{column}_box_violin.png")

phase_observations = []
for column in BIO_COLS:
    means = phase_mean[column].dropna()
    if len(means):
        highest, lowest = means.idxmax(), means.idxmin()
        phase_observations.append(
            f"{column} phase means span {means.min():.3f}–{means.max():.3f} "
            f"(highest in {highest}, lowest in {lowest}; descriptive difference {means.max() - means.min():.3f})."
        )
report_findings["phase"] = " ".join(phase_observations) + " These are unadjusted record-level means; differences are descriptive, not causal or inferential evidence."
print(report_findings["phase"])

### Findings

Compare the phase means alongside within-phase spread in the boxplots and violins. Any phase differences are observed associations in these records and do not establish causation; row-level summaries also do not account for repeated observations per participant.

## 7. Visualization Export

All plots generated above are saved in `reports/figures/`. The inventory below is also included in the generated Phase 3 report.

In [ ]:
figure_inventory = pd.DataFrame({"file": plot_files})
display(figure_inventory)
print(f"Exported {len(plot_files)} figures to {FIG_DIR.relative_to(ROOT)}")
report_tables["figures"] = figure_inventory
report_findings["visualization"] = f"Exported {len(plot_files)} descriptive plots to reports/figures; filenames identify each measure or comparison."
print(report_findings["visualization"])

### Findings

The inventory lists each exported plot. Together they provide categorical counts, univariate distributions, outlier context, and phase-stratified comparisons.

## 8. Phase 3 Report

Assemble the notebook’s computed tables and plain-English findings into `reports/eda3_summary.md`. The report is regenerated when this cell is run.

In [ ]:
def report_table(title, table, index=True):
    return f"### {title}\n\n{markdown_table(table, index=index)}"

phase_mismatch = int((df["phase"].notna() & df["phase.1"].notna() & df["phase"].ne(df["phase.1"])).sum())
heavy_proxy_above_one = int((pd.to_numeric(df["heavy_bleeding_flag_proxy"], errors="coerce") > 1).sum())
overview = pd.DataFrame(
    [
        ("Rows (day-level records)", f"{len(df):,}"),
        ("Columns", f"{df.shape[1]:,}"),
        ("Participants (`id`)", f"{df['id'].nunique():,}" if "id" in df else "not available"),
        ("Required columns present", f"{len(REQUIRED_COLS) - len(missing_columns)} / {len(REQUIRED_COLS)}"),
        ("Phase / phase.1 non-null disagreements", f"{phase_mismatch:,}"),
        ("heavy_bleeding_flag_proxy values > 1", f"{heavy_proxy_above_one:,}"),
    ], columns=["metric", "value"]
)

data_quality = pd.DataFrame(
    [
        ("Validation", report_findings["validation"]),
        ("Proxy encodings", report_findings["flags"]),
        ("Temperature metadata / ranges", report_findings["temperature"]),
        ("Phase-label agreement", f"{phase_mismatch:,} records have nonmissing phase and phase.1 labels that disagree."),
    ], columns=["area", "observation"]
)
recommendations = pd.DataFrame(
    {
        "recommended next EDA step": [
            "Confirm temperature units, sensor semantics, and the meaning of `type` before interpreting temperature values physiologically.",
            "Document how `phase` and `phase.1` were generated; resolve missing or discrepant labels before phase-stratified comparisons.",
            "Trace the construction and intended interpretation of `heavy_bleeding_flag_proxy` and `HMB_label`; do not apply cutoffs until specified.",
            "Review records with sparse temperature sampling and IQR-flagged extremes against source-level acquisition context.",
            "Describe within-participant and per-cycle data coverage and variability, accounting for repeated day-level records; keep the next phase descriptive.",
        ]
    }
)
figure_report = pd.DataFrame({
    "plot file": plot_files,
    "report link": [f"[PNG](figures/{filename})" for filename in plot_files],
})

report_parts = [
    "# EDA Phase 3 Summary — Menstrual Physiology and Hormonal Signals",
    "",
    "**Project:** CycleInsight — Menstrual Intelligence System  ",
    "**Dataset:** `datasets/cleaned_dataset.csv`  ",
    "**Analysis grain:** day-level records; repeated records from a participant are not independent.  ",
    "**Scope:** descriptive EDA only. No machine learning, feature engineering, predictive modeling, causal claims, or clinical diagnosis.",
    "",
    "## Dataset Overview",
    report_table("Overview", overview, index=False),
    "",
    "## 1. Data Validation",
    report_table("Requested Columns, Dtypes, and Missingness", report_tables["validation"]),
    "",
    report_table("Observed Categorical Levels", report_tables["categories"], index=False),
    "",
    report_table("Configured Suspicious-Value Checks", report_tables["suspicious"], index=False),
    "",
    f"### Findings\n\n{report_findings['validation']} These checks are exploratory and do not define clinical normality.",
    "",
    "## 2. Menstrual Feature Analysis",
    report_table("Categorical Frequencies and Percentages", report_tables["menstrual_categories"], index=False),
    "",
    report_table("Numeric Summary Statistics", report_tables["menstrual_numeric_summary"].round(3)),
    "",
    report_table("Numeric IQR Outlier Counts", report_tables["menstrual_iqr"].round(3)),
    "",
    f"### Findings\n\n{report_findings['menstrual']}",
    "",
    "## 3. Flag Analysis",
    report_table("Observed Flag and Score Encodings", report_tables["flags"].round(3)),
    "",
    f"### Findings\n\n{report_findings['flags']}",
    "",
    "## 4. Hormonal and Primary Temperature Analysis",
    report_table("Hormone Summary Statistics", report_tables["hormone_summary"].round(4)),
    "",
    report_table("Hormone IQR Outlier Counts", report_tables["hormone_iqr"].round(4)),
    "",
    f"### Findings\n\n{report_findings['hormones']}",
    "",
    "## 5. Temperature Feature Analysis",
    report_table("Temperature Feature Summary Statistics", report_tables["temperature_summary"].round(4)),
    "",
    report_table("Temperature Feature IQR Outlier Counts", report_tables["temperature_iqr"].round(4)),
    "",
    f"### Findings\n\n{report_findings['temperature']}",
    "",
    "## 6. Biological Consistency Checks",
    report_table("Mean LH, Estrogen, and BBT by Phase", report_tables["phase_means"].round(4)),
    "",
    report_table("Phase-Stratified Count, Mean, Median, and Standard Deviation", report_tables["phase_stats"].round(4)),
    "",
    f"### Findings\n\n{report_findings['phase']}",
    "",
    "## 7. Visualization Export",
    report_table("Exported Figures", figure_report, index=False),
    "",
    f"### Findings\n\n{report_findings['visualization']}",
    "",
    "## Data Quality Observations",
    report_table("Quality Notes", data_quality, index=False),
    "",
    "## Recommendations for the Next EDA Phase",
    report_table("Recommended Follow-up", recommendations, index=False),
    "",
    "## Findings",
    "The analyses describe observed distributions and phase-group differences only. They do not establish causation, diagnostic thresholds, or independent participant-level effects. Confirm field definitions and temperature units before physiological interpretation.",
    "",
    "*Generated by `notebooks/eda_03_menstrual_hormonal.ipynb`.*",
]
REPORT_PATH.write_text("\n".join(report_parts) + "\n", encoding="utf-8")
print(f"Wrote {REPORT_PATH.relative_to(ROOT)}")
print(f"Exported {len(plot_files)} plot files to {FIG_DIR.relative_to(ROOT)}")

### Findings

The generated summary contains the requested descriptive tables, phase observations, quality notes, and figure links. Interpretations remain descriptive and should be revisited after field definitions and sensor metadata are confirmed.